# <font color="#418FDE" size="6.5" uppercase>**Workflow Design**</font>

>Last update: 20260610.
    
By the end of this Lecture, you will be able to:
- Design an end-to-end workflow for a defined medical data task using built-in Python components. 
- Organize code into functions and main execution steps that are easy to read and maintain. 
- Document workflow assumptions, required inputs, and expected outputs for reproducible use. 


## **1. Planning Clinical Workflows**

### **1.1. Define Data Goals**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_A/image_01_01.jpg?v=1781148198" width="250">



>* Set a precise clinical data goal
>* Use goals to guide workflow decisions

>* Turn clinical ideas into data operations
>* Use basic Python structures thoughtfully

>* Make assumptions and data context explicit
>* Define workflow limits for reproducible use



In [ ]:
#@title Python Code - Define Data Goals

# Define goals before clinical processing.
# Use functions for readable workflows.
# Document assumptions and expected outputs.

# Create a tiny clinical dataset.
records = [
    {"patient_id": "A001", "condition": "diabetes", "last_a1c_days": 210},
    {"patient_id": "A002", "condition": "hypertension", "last_a1c_days": None},

    {"patient_id": "A003", "condition": "diabetes", "last_a1c_days": 95},
    {"patient_id": "A004", "condition": "diabetes", "last_a1c_days": 370},
]

# Store the workflow goal clearly.
goal = {
    "question": "Which diabetes patients are overdue for A1c monitoring?",
    "population": "Patients with diabetes in the clinic list.",

    "required_fields": ["patient_id", "condition", "last_a1c_days"],
    "expected_output": "Patient identifiers needing A1c follow-up.",
}

# Check required fields before analysis.
def validate_records(records, required_fields):
    missing_messages = []

    for index, record in enumerate(records):
        missing = [field for field in required_fields if field not in record]

        if missing:
            message = f"Row {index} missing {missing}"
            missing_messages.append(message)

    return missing_messages

# Flag patients using stated assumptions.
def find_overdue_patients(records, threshold_days):
    flagged = []

    for record in records:
        has_diabetes = record["condition"] == "diabetes"
        days = record["last_a1c_days"]

        if has_diabetes and days is not None and days > threshold_days:
            flagged.append(record["patient_id"])

    return flagged

# Run the planned workflow steps.
def main():
    threshold_days = 180
    problems = validate_records(records, goal["required_fields"])

    if problems:
        print("Validation failed:", problems[0])
        return None

    overdue = find_overdue_patients(records, threshold_days)
    print("Clinical question:", goal["question"])

    print("Assumption: overdue means more than", threshold_days, "days.")
    print("Expected output:", goal["expected_output"])

    print("Patients flagged:", ", ".join(overdue) if overdue else "None")
    return overdue

# Execute the workflow demonstration.
result = main()



### **1.2. Mapping Workflow Steps**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_A/image_01_02.jpg?v=1781148200" width="250">



>* Turn clinical goals into ordered workflow steps
>* Handle messy data before reliable outputs

>* Prepare data before clinical analysis
>* Order steps to avoid misleading results

>* Add checkpoints to catch data problems early
>* Define outputs for reproducible clinical workflows



In [ ]:
#@title Python Code - Mapping Workflow Steps

# Map a clinical workflow clearly.
# Use tiny built-in patient data.
# Check assumptions before analysis.

from datetime import date, datetime

# Define required workflow inputs.
required_fields = ["patient_id", "discharge_date", "next_admit_date"]

# Create small discharge records.
records = [
    {"patient_id": "P001", "discharge_date": "2024-01-01", "next_admit_date": "2024-01-20"},
    {"patient_id": "P002", "discharge_date": "2024-01-03", "next_admit_date": "2024-02-15"},
    {"patient_id": "P003", "discharge_date": "2024-01-05", "next_admit_date": ""},
]

# Step one validates required fields.
def validate_records(data, required):
    if not data:
        return False, "No records were supplied."
    missing = [field for field in required if field not in data[0]]
    if missing:
        return False, "Missing fields: " + ", ".join(missing)
    return True, "All required fields are present."

# Step two converts date strings safely.
def parse_date(value):
    if not value:
        return None
    return datetime.strptime(value, "%Y-%m-%d").date()

# Step three cleans records for analysis.
def clean_records(data):
    cleaned = []
    for row in data:
        new_row = dict(row)
        new_row["discharge_date"] = parse_date(row["discharge_date"])
        new_row["next_admit_date"] = parse_date(row["next_admit_date"])
        cleaned.append(new_row)
    return cleaned

# Step four applies clinical logic.
def flag_readmissions(data, window_days=30):
    results = []
    for row in data:
        days = None
        readmitted = False
        if row["next_admit_date"] and row["discharge_date"]:
            days = (row["next_admit_date"] - row["discharge_date"]).days
            readmitted = 0 <= days <= window_days
        results.append((row["patient_id"], days, readmitted))
    return results

# Main execution follows the workflow map.
def main():
    ok, message = validate_records(records, required_fields)
    print("Checkpoint 1:", message)
    if not ok:
        return
    cleaned = clean_records(records)
    results = flag_readmissions(cleaned)
    total = len(results)
    flagged = sum(1 for item in results if item[2])
    print("Checkpoint 2: cleaned records =", len(cleaned))
    print("Workflow output: readmissions flagged =", flagged, "of", total)
    print("Expected output: patient-level readmission flags.")

main()



### **1.3. Workflow Logic**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_A/image_01_03.jpg?v=1781148202" width="250">



>* Connect clinical goals to workflow decisions
>* Plan readable logic with built-in Python

>* Use decisions to handle messy clinical data
>* Make workflow choices transparent and defensible

>* Track data from input to output
>* Add checks for reusable, clear results



In [ ]:
#@title Python Code - Workflow Logic

# Plan workflow logic before coding.
# Define assumptions, inputs, and outputs.
# Use functions for maintainable steps.

from datetime import datetime, timedelta
from pprint import pprint

# Create tiny encounter records.
encounters = [
    {"patient_id": "P01", "date": "2024-01-01", "type": "discharge"},
    {"patient_id": "P01", "date": "2024-01-18", "type": "admission"},
    {"patient_id": "P02", "date": "2024-02-01", "type": "discharge"},
    {"patient_id": "", "date": "2024-02-10", "type": "admission"},
    {"patient_id": "P02", "date": "bad-date", "type": "admission"},
]

# Define expected workflow assumptions.
required_fields = {"patient_id", "date", "type"}
readmission_window = timedelta(days=30)
allowed_types = {"admission", "discharge"}

# Parse one date safely.
def parse_date(date_text):
    try:
        return datetime.strptime(date_text, "%Y-%m-%d").date()
    except ValueError:
        return None

# Validate one clinical record.
def validate_record(record):
    if not required_fields.issubset(record):
        return False, "missing required field"
    if not record["patient_id"]:
        return False, "missing patient identifier"
    if record["type"] not in allowed_types:
        return False, "unexpected encounter type"
    if parse_date(record["date"]) is None:
        return False, "unreadable date"
    return True, "accepted"

# Separate usable records from review records.
def clean_records(records):
    clean, review = [], []
    for record in records:
        valid, reason = validate_record(record)
        if valid:
            record = record.copy()
            record["parsed_date"] = parse_date(record["date"])
            clean.append(record)
        else:
            review.append({"record": record, "reason": reason})
    return clean, review

# Detect readmissions within the defined window.
def find_readmissions(clean):
    results = []
    discharges = [r for r in clean if r["type"] == "discharge"]
    admissions = [r for r in clean if r["type"] == "admission"]
    for discharge in discharges:
        match_found = False
        for admission in admissions:
            same_patient = admission["patient_id"] == discharge["patient_id"]
            days_after = admission["parsed_date"] - discharge["parsed_date"]
            if same_patient and timedelta(days=0) < days_after <= readmission_window:
                match_found = True
        results.append({"patient_id": discharge["patient_id"], "readmitted_30d": match_found})
    return results

# Run the workflow in clear steps.
clean_records_list, review_records = clean_records(encounters)
workflow_output = find_readmissions(clean_records_list)

# Print concise reproducibility notes.
print("Task: detect thirty-day readmissions after discharge.")
print("Required inputs: patient_id, date, and type.")
print(f"Accepted records: {len(clean_records_list)}")
print(f"Records needing review: {len(review_records)}")
print("Expected output: one row per discharge.")
print("Workflow output:")
pprint(workflow_output)



## **2. Readable Script Structure**

### **2.1. Function First Organization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_A/image_02_01.jpg?v=1781148192" width="250">



>* Place reusable functions before workflow execution
>* Named steps improve readability, testing, and reuse

>* Give each function one clear responsibility
>* Localize changes and simplify error tracing

>* Visible logic supports collaboration and reproducibility
>* Named functions simplify auditing and adaptation



In [ ]:
#@title Python Code - Function First Organization

# This script demonstrates function first workflow organization.
# Small clinical data supports readable workflow structure.
# The main section appears after reusable functions.

import csv
from io import StringIO

# Define required columns for the workflow.
REQUIRED_COLUMNS = ["patient_id", "visit_date", "systolic"]

# Store tiny example data inside the script.
RAW_DATA = """patient_id,visit_date,systolic\nA001,2026-01-02,138\nA001,2026-01-02,138\nA002,01/03/2026,151\nA003,2026-01-04,\nA004,2026-01-05,124\n"""

# Load patient rows from comma separated text.
def load_patient_rows(csv_text):
    reader = csv.DictReader(StringIO(csv_text))
    rows = list(reader)
    return rows

# Confirm expected input columns are present.
def validate_required_columns(rows, required_columns):
    if not rows:
        raise ValueError("No rows were provided.")
    missing = set(required_columns) - set(rows[0].keys())
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    return True

# Remove repeated patient visit records.
def remove_duplicate_visits(rows):
    seen_keys = set()
    cleaned_rows = []
    for row in rows:
        key = (row["patient_id"], row["visit_date"])
        if key not in seen_keys:
            cleaned_rows.append(row)
            seen_keys.add(key)
    return cleaned_rows

# Keep rows with usable systolic values.
def keep_complete_systolic_rows(rows):
    complete_rows = []
    for row in rows:
        if row["systolic"].strip().isdigit():
            row["systolic"] = int(row["systolic"])
            complete_rows.append(row)
    return complete_rows

# Summarize cleaned blood pressure records.
def summarize_systolic(rows):
    values = [row["systolic"] for row in rows]
    count = len(values)
    average = sum(values) / count if count else 0
    high_count = sum(value >= 140 for value in values)
    return count, average, high_count

# Run the workflow using named functions.
def main():
    rows = load_patient_rows(RAW_DATA)
    validate_required_columns(rows, REQUIRED_COLUMNS)
    unique_rows = remove_duplicate_visits(rows)
    clean_rows = keep_complete_systolic_rows(unique_rows)

    count, average, high_count = summarize_systolic(clean_rows)
    print("Readable workflow demonstration")
    print(f"Input rows: {len(rows)}")
    print(f"Clean rows: {count}")
    print(f"Average systolic: {average:.1f} mmHg")
    print(f"High systolic records: {high_count}")

# Execute the workflow only after definitions.
main()



### **2.2. Main Execution Flow**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_A/image_02_02.jpg?v=1781148195" width="250">



>* Show the workflow from input to output
>* Keep detailed logic inside named functions

>* Centralize workflow control for easier maintenance
>* Coordinate functions so changes stay safer

>* Show assumptions through clear workflow decisions
>* Support reproducible review, auditing, and adaptation



In [ ]:
#@title Python Code - Main Execution Flow

# Demonstrate a clear medical workflow.
# Keep orchestration separate from details.
# Print concise reproducible workflow outputs.

import csv
import tempfile
from pathlib import Path

# Create a tiny clinical source file.
def create_sample_visit_file(folder_path):
    file_path = Path(folder_path) / "ed_visits.csv"
    rows = [
        {"patient_id": "P001", "arrival_hour": "8", "wait_minutes": "34", "disposition": "home"},
        {"patient_id": "P002", "arrival_hour": "9", "wait_minutes": "82", "disposition": "admit"},

        {"patient_id": "P003", "arrival_hour": "9", "wait_minutes": "15", "disposition": "home"},
        {"patient_id": "P004", "arrival_hour": "10", "wait_minutes": "105", "disposition": "admit"},
    ]

    field_names = ["patient_id", "arrival_hour", "wait_minutes", "disposition"]
    with open(file_path, "w", newline="") as file_handle:
        writer = csv.DictWriter(file_handle, fieldnames=field_names)
        writer.writeheader()

        writer.writerows(rows)
    return file_path

# Load rows from the source file.
def load_visit_rows(file_path):
    with open(file_path, newline="") as file_handle:
        reader = csv.DictReader(file_handle)
        loaded_rows = list(reader)

    return loaded_rows

# Confirm required fields are present.
def validate_visit_rows(rows):
    required_fields = {"patient_id", "arrival_hour", "wait_minutes", "disposition"}
    if not rows:
        raise ValueError("No visit rows were loaded.")

    available_fields = set(rows[0].keys())
    missing_fields = required_fields - available_fields
    if missing_fields:
        raise ValueError(f"Missing fields: {sorted(missing_fields)}")

# Convert text fields into useful values.
def transform_visit_rows(rows):
    transformed_rows = []
    for row in rows:
        wait_minutes = int(row["wait_minutes"])

        transformed_row = dict(row)
        transformed_row["wait_minutes"] = wait_minutes
        transformed_row["long_wait"] = wait_minutes >= 60
        transformed_rows.append(transformed_row)

    return transformed_rows

# Summarize the transformed visit data.
def calculate_dashboard_metrics(rows):
    total_visits = len(rows)
    admit_count = sum(row["disposition"] == "admit" for row in rows)
    long_wait_count = sum(row["long_wait"] for row in rows)

    average_wait = sum(row["wait_minutes"] for row in rows) / total_visits
    metrics = {
        "total_visits": total_visits,
        "admit_count": admit_count,

        "long_wait_count": long_wait_count,
        "average_wait": round(average_wait, 1),
    }
    return metrics

# Save a compact report for reuse.
def save_dashboard_report(metrics, folder_path):
    report_path = Path(folder_path) / "weekly_ed_summary.txt"
    report_lines = [
        f"Total visits: {metrics['total_visits']}",
        f"Admissions: {metrics['admit_count']}",

        f"Long waits: {metrics['long_wait_count']}",
        f"Average wait minutes: {metrics['average_wait']}",
    ]
    report_path.write_text("\n".join(report_lines))

    return report_path

# Coordinate the workflow from input to output.
def run_workflow():
    with tempfile.TemporaryDirectory() as temp_folder:
        source_path = create_sample_visit_file(temp_folder)
        raw_rows = load_visit_rows(source_path)

        validate_visit_rows(raw_rows)
        clean_rows = transform_visit_rows(raw_rows)
        metrics = calculate_dashboard_metrics(clean_rows)
        report_path = save_dashboard_report(metrics, temp_folder)

        print("Main flow completed: load, validate, transform, summarize, save.")
        print(f"Rows processed: {metrics['total_visits']}")
        print(f"Average wait minutes: {metrics['average_wait']}")
        print(f"Report saved as: {report_path.name}")

# Run the workflow from one predictable place.
run_workflow()



### **2.3. Clear Main Flow**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_A/image_02_03.jpg?v=1781148196" width="250">



>* Show the workflow in a clear sequence
>* Keep details inside well-named functions

>* Show workflow order in the main script
>* Use functions for details and safer collaboration

>* Simple main steps support reliable maintenance
>* Clear structure helps workflows adapt over time



In [ ]:
#@title Python Code - Clear Main Flow

# Demonstrate a readable medical workflow.
# Keep the main flow clearly visible.
# Use functions for detailed workflow steps.

import csv
from statistics import mean
from tempfile import NamedTemporaryFile

# Define small patient records for demonstration.
records = [
    {"patient_id": "P001", "systolic": "128", "diastolic": "82"},
    {"patient_id": "P002", "systolic": "", "diastolic": "78"},

    {"patient_id": "P003", "systolic": "146", "diastolic": "91"},
    {"patient_id": "P004", "systolic": "118", "diastolic": "76"},
]

# Create a tiny temporary CSV file.
def create_input_file(input_records):
    fieldnames = ["patient_id", "systolic", "diastolic"]
    temp_file = NamedTemporaryFile(mode="w", delete=False, newline="")

    writer = csv.DictWriter(temp_file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(input_records)

    temp_file.close()
    return temp_file.name

# Load records from the input file.
def load_patient_records(file_path):
    with open(file_path, newline="") as opened_file:
        reader = csv.DictReader(opened_file)
        loaded_records = list(reader)

    return loaded_records

# Check required columns before analysis.
def validate_records(loaded_records):
    required_columns = {"patient_id", "systolic", "diastolic"}
    available_columns = set(loaded_records[0].keys())

    if not required_columns.issubset(available_columns):
        raise ValueError("Required columns are missing.")

    return len(loaded_records) > 0

# Remove incomplete rows and convert numbers.
def clean_records(loaded_records):
    cleaned_records = []
    for record in loaded_records:
        if record["systolic"] and record["diastolic"]:
            cleaned_records.append({
                "patient_id": record["patient_id"],
                "systolic": int(record["systolic"]),

                "diastolic": int(record["diastolic"]),
            })

    return cleaned_records

# Create simple clinical summary values.
def summarize_blood_pressure(cleaned_records):
    systolic_values = [record["systolic"] for record in cleaned_records]
    high_count = sum(value >= 140 for value in systolic_values)

    return {
        "included_patients": len(cleaned_records),
        "mean_systolic": round(mean(systolic_values), 1),
        "high_systolic_count": high_count,
    }

# Report only concise workflow results.
def report_summary(summary):
    print("Clear main flow demonstration")
    print(f"Included patients: {summary['included_patients']}")
    print(f"Mean systolic pressure: {summary['mean_systolic']} mmHg")

    print(f"High systolic count: {summary['high_systolic_count']}")

# Run the workflow in readable order.
def main():
    input_path = create_input_file(records)
    raw_records = load_patient_records(input_path)

    validate_records(raw_records)
    prepared_records = clean_records(raw_records)
    summary = summarize_blood_pressure(prepared_records)

    report_summary(summary)

# Keep execution separate from definitions.
if __name__ == "__main__":
    main()



## **3. Reproducibility Notes**

### **3.1. Document Data Assumptions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_A/image_03_01.jpg?v=1781148189" width="250">



>* State expected data structure before processing
>* Prevent silent errors and misleading results

>* State content, quality, and context assumptions
>* Make workflows easier to evaluate and adapt

>* Define limits for interpreting workflow results
>* Document assumptions for reproducible, responsible reuse



In [ ]:
#@title Python Code - Document Data Assumptions

# Reproducibility notes make workflow assumptions explicit.
# Small checks can reveal hidden data problems.
# This example documents expectations before analysis.

import pandas as pd

# Create a tiny deidentified encounter dataset.
records = [
    {"encounter_id": "E001", "patient_id": "P10", "visit_date": "2026-01-04", "systolic_bp_mmHg": 132},
    {"encounter_id": "E002", "patient_id": "P11", "visit_date": "2026-01-05", "systolic_bp_mmHg": 118},
    {"encounter_id": "E003", "patient_id": "P10", "visit_date": "2026-01-09", "systolic_bp_mmHg": 141},
]

# Convert records into a small table.
ed_visits = pd.DataFrame(records)

# Store plain language data assumptions.
assumptions = {
    "row_unit": "Each row is one emergency department encounter.",
    "date_format": "visit_date uses ISO year-month-day text.",
    "identifier_status": "patient_id is deidentified before analysis.",
    "blood_pressure_unit": "Systolic blood pressure is measured in mmHg.",
}

# Define required columns for this workflow.
required_columns = [
    "encounter_id",
    "patient_id",
    "visit_date",
    "systolic_bp_mmHg",
]

# Check whether required columns are present.
missing_columns = [
    name for name in required_columns
    if name not in ed_visits.columns
]

# Check whether encounter identifiers are unique.
unique_encounters = ed_visits["encounter_id"].is_unique

# Check whether visit dates parse safely.
parsed_dates = pd.to_datetime(
    ed_visits["visit_date"],
    errors="coerce",
)

# Summarize assumption checks as short notes.
check_notes = [
    f"Missing required columns: {missing_columns or 'none'}",
    f"Encounter IDs unique: {unique_encounters}",
    f"Unparseable visit dates: {parsed_dates.isna().sum()}",
]

# Print concise reproducibility notes.
print("Documented data assumptions:")
for key, note in assumptions.items():
    print(f"- {key}: {note}")

# Print concise validation results.
print("Validation checks:")
for note in check_notes:
    print(f"- {note}")



### **3.2. Required Inputs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_A/image_03_02.jpg?v=1781148191" width="250">



>* List every needed workflow input clearly
>* Prevent misinterpretation and support confident reuse

>* Describe each input’s format and meaning
>* Explain why inputs affect valid results

>* Record practical data assumptions before running workflows
>* Prevent errors and support reuse across teams



In [ ]:
#@title Python Code - Required Inputs

# Document inputs before workflow execution.
# Validate required medical data fields.
# Report assumptions for reproducible reuse.

import pandas as pd

# Define required workflow inputs clearly.
required_inputs = {
    "admissions_file": "CSV table of hospital admissions",
    "patient_id_column": "Stable patient identifier across visits",
    "admission_date_column": "Admission date in YYYY-MM-DD format",
    "discharge_date_column": "Discharge date in YYYY-MM-DD format",
}

# Create a tiny example admissions table.
admissions = pd.DataFrame({
    "patient_id": ["P001", "P002", "P001"],
    "admission_date": ["2024-01-02", "2024-01-05", "2024-01-20"],
    "discharge_date": ["2024-01-04", "2024-01-08", "2024-01-23"],
})

# Store workflow configuration assumptions.
configuration = {
    "patient_id_column": "patient_id",
    "admission_date_column": "admission_date",
    "discharge_date_column": "discharge_date",
    "readmission_window_days": 30,
}

# Check that configured columns exist.
def validate_required_inputs(dataframe, config):
    needed_columns = [
        config["patient_id_column"],
        config["admission_date_column"],
        config["discharge_date_column"],
    ]

    missing_columns = [
        column for column in needed_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        return False, missing_columns
    return True, []

# Print compact reproducibility notes.
print("Required inputs for this workflow:")
for key, meaning in required_inputs.items():
    print(f"- {key}: {meaning}")

# Validate inputs before processing.
is_valid, missing = validate_required_inputs(admissions, configuration)

# Report whether workflow can safely start.
if is_valid:
    print("Input check: PASS, all required columns are present.")
else:
    print(f"Input check: FAIL, missing columns: {missing}")

# Report key interpretation assumptions.
print("Assumption: patient_id is stable across admissions.")
print("Assumption: dates use YYYY-MM-DD and represent clinical events.")
print(f"Expected output: readmissions within {configuration['readmission_window_days']} days.")



### **3.3. Required Inputs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_A/image_03_03.jpg?v=1781148187" width="250">



>* List all files, settings, and context
>* Make workflows repeatable and outputs understandable

>* Specify each input’s structure and meaning
>* Clarify formats to prevent ambiguity

>* Record hidden dependencies needed for correct execution
>* Support preparation, troubleshooting, and repeatable results



In [ ]:
#@title Python Code - Required Inputs

# Reproducible workflows begin with documented required inputs.
# This example validates a tiny medical extract.
# Missing assumptions are caught before processing.

# No extra installs are required for this example.

# Define required input documentation as structured metadata.
required_inputs = {
    "patient_id": "De-identified patient identifier",
    "collection_date": "Specimen collection date in ISO format",
    "test_name": "Laboratory test name",
    "result_value": "Numeric laboratory result",
    "unit": "Expected measurement unit",
}

# Create a tiny example laboratory extract.
lab_rows = [
    {"patient_id": "P001", "collection_date": "2026-01-03", "test_name": "Hemoglobin", "result_value": 13.4, "unit": "g/dL"},
    {"patient_id": "P002", "collection_date": "2026-01-04", "test_name": "Hemoglobin", "result_value": 11.2, "unit": "g/dL"},
]

# State workflow assumptions near the validation logic.
expected_unit = "g/dL"
expected_test = "Hemoglobin"
minimum_rows = 1

# Validate required columns before analysis begins.
def validate_required_inputs(rows, required, test_name, unit_name):
    if len(rows) < minimum_rows:
        return False, "No laboratory rows were supplied."

    # Check every required field appears in every row.
    for row_index, row in enumerate(rows, start=1):
        missing = [name for name in required if name not in row]
        if missing:
            return False, f"Row {row_index} missing {missing}."

    # Check expected clinical context and measurement unit.
    for row_index, row in enumerate(rows, start=1):
        if row["test_name"] != test_name:
            return False, f"Row {row_index} has unexpected test."
        if row["unit"] != unit_name:
            return False, f"Row {row_index} has unexpected unit."

    # Confirm result values are numeric enough to summarize.
    for row_index, row in enumerate(rows, start=1):
        if not isinstance(row["result_value"], (int, float)):
            return False, f"Row {row_index} has nonnumeric result."
    return True, "All documented inputs passed validation."

# Run validation before producing any workflow output.
is_ready, validation_message = validate_required_inputs(
    lab_rows,
    required_inputs,
    expected_test,
    expected_unit,
)

# Summarize only when required inputs are acceptable.
if is_ready:
    values = [row["result_value"] for row in lab_rows]
    average_value = sum(values) / len(values)
else:
    average_value = None

# Print concise reproducibility notes for reviewers.
print("Required input fields:", ", ".join(required_inputs.keys()))
print("Assumed test and unit:", expected_test, expected_unit)
print("Validation result:", validation_message)
print("Rows received:", len(lab_rows))
print("Average result:", average_value)




# <font color="#418FDE" size="6.5" uppercase>**Workflow Design**</font>


In this lecture, you learned to:
- Design an end-to-end workflow for a defined medical data task using built-in Python components. 
- Organize code into functions and main execution steps that are easy to read and maintain. 
- Document workflow assumptions, required inputs, and expected outputs for reproducible use. 

<font color='yellow'>Congratulations on completing this course!</font>